# Solar Generation EDA — 4-Subplot Analysis

Exploratory data analysis for the **3.57 kW rooftop system in Karnal, Haryana**.  
Data covers **13 months (Apr 2025 – May 2026)** sourced from InfluxDB at 15-minute resolution.

**Panels:**
1. **Hourly Generation Boxplot** — hourly variance across all days
2. **Irradiance vs DC Power scatter** — coloured by ambient temperature
3. **Mean Power Heatmap** — hour × month pivot
4. **Daily Energy Line Plot** — with gap highlighting and 7-day rolling mean

In [ ]:
!pip install xgboost shap plotly influxdb-client --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.dpi'] = 150
print('Libraries loaded.')

In [ ]:
%run data_loader.py
df = load_solar_data()
print(f'Loaded {len(df):,} rows  |  Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── Ensure correct dtypes ──────────────────────────────────────────────────────
df['datetime'] = pd.to_datetime(df['datetime'])
df['date']     = pd.to_datetime(df['date'])
df['hour']     = df['hour'].astype(int)
df['month']    = df['month'].astype(int)

# Month order for heatmap x-axis
MONTH_ORDER = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Restrict to daylight hours (06:00-19:00) for boxplot and heatmap
df_day = df[(df['hour'] >= 6) & (df['hour'] <= 19)].copy()

print(f'Daylight rows: {len(df_day):,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BUILD 2x2 FIGURE
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'Solar Generation EDA — Karnal Rooftop 3.57 kW (Apr 2025 – May 2026)',
    fontsize=15, fontweight='bold', y=1.01
)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 1 (top-left): Hourly Generation Distribution — daily boxplot
# ─────────────────────────────────────────────────────────────────────────────
ax1 = axes[0, 0]

# Compute per-hour median for colour mapping
hour_medians = (
    df_day.groupby('hour')['power_now_w']
    .median()
    .reset_index()
    .rename(columns={'power_now_w': 'median_power'})
)
norm_p = mcolors.Normalize(
    vmin=hour_medians['median_power'].min(),
    vmax=hour_medians['median_power'].max()
)
cmap_p  = plt.cm.YlOrRd
palette = {
    int(row['hour']): cmap_p(norm_p(row['median_power']))
    for _, row in hour_medians.iterrows()
}

sns.boxplot(
    data=df_day,
    x='hour', y='power_now_w',
    order=list(range(6, 20)),
    palette=palette,
    linewidth=0.8,
    flierprops=dict(marker='.', markersize=2, alpha=0.3),
    ax=ax1
)

# Annotate median values above each box
for tick_pos, hr in enumerate(range(6, 20)):
    row = hour_medians[hour_medians['hour'] == hr]
    if not row.empty:
        med = row['median_power'].values[0]
        if med > 30:
            ax1.text(
                tick_pos, med + 25, f'{med:.0f}',
                ha='center', va='bottom', fontsize=6.5,
                color='darkred', fontweight='bold'
            )

ax1.set_title('Hourly Generation Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('AC Output Power (W)')
ax1.set_xticklabels([str(h) for h in range(6, 20)])
ax1.grid(axis='y', alpha=0.3)

# Colourbar for median power
sm1 = plt.cm.ScalarMappable(cmap=cmap_p, norm=norm_p)
sm1.set_array([])
plt.colorbar(sm1, ax=ax1, label='Median Power (W)', pad=0.01)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 2 (top-right): Irradiance vs DC Power — scatter coloured by temperature
# ─────────────────────────────────────────────────────────────────────────────
ax2 = axes[0, 1]

sub2 = df[['irradiance', 'dc_power', 'ambient_temp']].dropna()
sc2 = ax2.scatter(
    sub2['irradiance'], sub2['dc_power'],
    c=sub2['ambient_temp'], cmap='RdYlGn_r',
    alpha=0.3, s=8
)
plt.colorbar(sc2, ax=ax2, label='Temp (°C)', pad=0.01)

# Linear regression line
slope, intercept, r_val, _, _ = stats.linregress(
    sub2['irradiance'], sub2['dc_power']
)
x_line = np.linspace(sub2['irradiance'].min(), sub2['irradiance'].max(), 300)
ax2.plot(
    x_line, slope * x_line + intercept,
    color='navy', linewidth=1.8,
    label=f'Linear fit   R²={r_val**2:.3f}'
)
ax2.legend(fontsize=9)
ax2.set_title(
    'Irradiance vs DC Power (coloured by temperature)',
    fontsize=12, fontweight='bold'
)
ax2.set_xlabel('Irradiance (W/m²)')
ax2.set_ylabel('DC Power (W)')
ax2.grid(alpha=0.3)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 3 (bottom-left): Heatmap — mean power by hour x month
# ─────────────────────────────────────────────────────────────────────────────
ax3 = axes[1, 0]

months_present = [m for m in MONTH_ORDER if m in df_day['month_name'].unique()]

pivot = (
    df_day
    .groupby(['hour', 'month_name'])['power_now_w']
    .mean()
    .reset_index()
    .pivot(index='hour', columns='month_name', values='power_now_w')
    .reindex(index=list(range(6, 20)), columns=months_present)
)

sns.heatmap(
    pivot, ax=ax3,
    cmap='YlOrRd',
    annot=True, fmt='.0f',
    linewidths=0.5, linecolor='grey',
    cbar_kws={'label': 'Mean Power (W)'}
)
ax3.set_title('Mean Power by Hour × Month (W)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Month')
ax3.set_ylabel('Hour of Day')
ax3.tick_params(axis='x', rotation=45)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 4 (bottom-right): Daily energy production line plot
# ─────────────────────────────────────────────────────────────────────────────
ax4 = axes[1, 1]

# 15-min intervals → kWh:  W × (15/60) / 1000
daily = (
    df.groupby('date')
    .agg(
        energy_kwh=('power_now_w', lambda x: (x * (15 / 60) / 1000).sum()),
        reading_count=('power_now_w', 'count')
    )
    .reset_index()
    .sort_values('date')
)
daily['date'] = pd.to_datetime(daily['date'])

# 7-day rolling mean
daily['rolling_7'] = daily['energy_kwh'].rolling(7, min_periods=3).mean()

# Days with < 10 readings  → data gap
gap_days = daily[daily['reading_count'] < 10]

# Shade gap days with red vertical spans
for _, grow in gap_days.iterrows():
    ax4.axvspan(
        grow['date'] - pd.Timedelta(hours=12),
        grow['date'] + pd.Timedelta(hours=12),
        color='red', alpha=0.25, lw=0
    )

ax4.plot(
    daily['date'], daily['energy_kwh'],
    color='steelblue', linewidth=0.9, alpha=0.75, label='Daily Energy (kWh)'
)
ax4.plot(
    daily['date'], daily['rolling_7'],
    color='darkorange', linewidth=2.0, linestyle='--', label='7-day Rolling Mean'
)

# Build legend including gap patch
gap_patch = mpatches.Patch(
    facecolor='red', alpha=0.25,
    label=f'Data gap  (n={len(gap_days)} days)'
)
handles, labels = ax4.get_legend_handles_labels()
ax4.legend(handles=handles + [gap_patch], fontsize=9)

ax4.set_title('Daily Energy Production (kWh)', fontsize=12, fontweight='bold')
ax4.set_xlabel('Date')
ax4.set_ylabel('Energy (kWh)')
ax4.tick_params(axis='x', rotation=30)
ax4.grid(alpha=0.3)

# ── Finalise and save ─────────────────────────────────────────────────────────
plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_analysis.png')

## Key Observations

| Panel | Insight |
|-------|---------|
| **Hourly Boxplot** | Peak output occurs 11:00–14:00. The wide IQR during midday reflects cloud-cover variability. Boxes are colour-coded by median — darker red = higher median output. |
| **Irradiance scatter** | Strong linear R² relationship. Higher ambient temperatures produce marginally lower DC power at the same irradiance — panel thermal de-rating is visible. |
| **Hour × Month heatmap** | Summer months (Apr–Jun) show highest peak watts; winter months show shorter generation windows. Each cell value is mean AC watts across that hour/month combination. |
| **Daily energy** | Red vertical spans mark days with < 10 readings (inverter downtime or communication loss). The 7-day rolling mean clearly reveals seasonal variation. |